## Sentiment Analysis

In this exercise we use the IMDb-dataset, which we will use to perform a sentiment analysis. The code below assumes that the data is placed in the same folder as this notebook. We see that the reviews are loaded as a pandas dataframe, and print the beginning of the first few reviews.

In [129]:
import numpy as np
import pandas as pd

reviews = pd.read_csv('reviews.txt', header=None)
labels = pd.read_csv('labels.txt', header=None)
Y = (labels=='positive').astype(np.int_)

print(type(reviews))
print(reviews.head())

<class 'pandas.core.frame.DataFrame'>
                                                   0
0  bromwell high is a cartoon comedy . it ran at ...
1  story of a man who has unnatural feelings for ...
2  homelessness  or houselessness as george carli...
3  airport    starts as a brand new luxury    pla...
4  brilliant over  acting by lesley ann warren . ...


**(a)** Split the reviews and labels in test, train and validation sets. The train and validation sets will be used to train your model and tune hyperparameters, the test set will be saved for testing. Use the `CountVectorizer` from `sklearn.feature_extraction.text` to create a Bag-of-Words representation of the reviews. Only use the 10,000 most frequent words (use the `max_features`-parameter of `CountVectorizer`).

**(b)** Explore the representation of the reviews. How is a single word represented? How about a whole review?

**(c)** Train a neural network with a single hidden layer on the dataset, tuning the relevant hyperparameters to optimize accuracy. 

**(d)** Test your sentiment-classifier on the test set.

**(e)** Use the classifier to classify a few sentences you write yourselves. 

## Imports

In [136]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

# Part A

In [138]:
# split data into train, validation, and test sets
X_temp, X_test, Y_temp, Y_test = train_test_split(reviews[0], Y, test_size=0.2, random_state=42)
X_train, X_val, Y_train, Y_val = train_test_split(X_temp, Y_temp, test_size=0.25, random_state=42)

# flatten the label arrays
Y_train = Y_train.values.flatten()
Y_val = Y_val.values.flatten()
Y_test = Y_test.values.flatten()

# convert text to bag of words
vectorizer = CountVectorizer(max_features=10000)
X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec = vectorizer.transform(X_val)
X_test_vec = vectorizer.transform(X_test)

# show how the data was split
print("Training set size:", X_train_vec.shape)
print("Validation set size:", X_val_vec.shape)
print("Test set size:", X_test_vec.shape)

# show how a single review looks before and after vectorization
print("\nExample original review:\n", X_train.iloc[0])
print("\nVectorized (Bag-of-Words) form:\n", X_train_vec[0])

Training set size: (15000, 10000)
Validation set size: (5000, 10000)
Test set size: (5000, 10000)

Example original review:
   birth of the beatles   for being a us television movie  released in the fall of     has actually been  so far the best movie which tells the tale of the the four lads from liverpool that revolutionized the music industry and the world . as told by the point of view of former beatle pete best . the performance from the entire cast is excellent but  most especially the performance by stephen mackenna as john lennon and rod culbertson as paul mccartney . the film was produced by a legend of the rock and roll era  mr dick clark . who a year earlier in     had produced another tv movie  that has stood the test of time starring  kurt rusell  in the lead role about another musical legend  elvis  . that movie was directed by an unknown director named  john carpenter  who went on to direct other successful movies such as  halloween    escape from new york   and  the thi

### Bag-of-Words

We have the Vectorized (Bag-of-Words) reviews, but what does it mean? What is BoW?

It's a way of turning text into numbers.
- We create a vocabulary of the most common words (in our case, the top 10 000 words).
- Then each review becomes a **vector** of size **10 000**, where each position counts **how many times a specific word appeared** in that review.

#### So let's say we have:

(0, 884)	3\
(0, 6174)	17\
(0, 8958)	35

**0** -> index of the review (we are looking at the first review in our dataset)\
**884** -> index of a word in the vocabulary\
**3** -> that word appeared 3 times in the review

So if we would like to represent it in one sentence then:
- **In review 0, the word at vocab index 884 appears 3 times.**

And if you're curious what is the word with index 884, here you go. :)

In [140]:
# get the word-to-index mapping
word2idx = vectorizer.vocabulary_

# reverse it to index-to-word
idx2word = {i: w for w, i in word2idx.items()}

# get word for index 884, 6174, and 8958
print("Word at index 884:", idx2word[884])
print("Word at index 6174:", idx2word[6174])
print("Word at index 8958:", idx2word[8958])

Word at index 884: birth
Word at index 6174: of
Word at index 8958: the


# Part B

### Representation of a Single Word in BoW
I already explained this in part A, but to quickly recap:

In the Bag-of-Words representation, a single word is just a number at a specific index in the 10 000 length vector. That number tells you how many times that word appeared in the review.

So if you see:

(0, 884)	3

It means that in review 0, the word at vocab index 884 appears 3 times, and we can even decode which word it was using the vocabulary mapping (like we did earlier with index 884 = "birth"). :)

### Representation of a Whole Review
A whole review is turned into a list with 10,000 numbers. Each number shows how many times a certain word from our vocabulary shows up in the review.

Most of the values are zero, because each review usually contains only a few words of the full vocabulary.

Let’s look at an example from our training data:

In [144]:
vec = X_train_vec[0].toarray()[0]

for idx, count in enumerate(vec):
    if count > 0:
        print(f"{idx2word[idx]}: {count}")

# we are skipping the words that don't appear in the review (count > 0) to save space and make it more clean

about: 1
actually: 1
among: 1
an: 1
and: 9
another: 2
appear: 1
arrive: 1
as: 5
at: 1
back: 1
be: 1
beat: 1
beatles: 3
been: 1
beginning: 1
being: 1
best: 2
birth: 3
blockbusters: 1
but: 1
by: 4
can: 1
care: 1
carpenter: 1
cast: 1
charming: 1
clark: 1
critics: 1
dick: 1
did: 3
direct: 2
directed: 1
director: 2
earlier: 1
ed: 1
edge: 1
elvis: 1
ends: 1
entire: 1
era: 1
escape: 1
especially: 1
essence: 1
excellent: 1
eye: 1
fall: 1
far: 1
film: 4
for: 4
former: 1
four: 2
from: 3
gives: 1
had: 2
halloween: 1
hardships: 1
has: 2
have: 1
he: 1
highly: 1
however: 1
in: 8
industry: 1
is: 3
it: 2
jedi: 1
john: 2
know: 1
kurt: 1
lead: 1
legend: 3
lennon: 1
life: 1
long: 1
many: 1
most: 1
movie: 4
movies: 1
mr: 2
music: 1
musical: 1
named: 1
needle: 1
new: 1
nor: 1
not: 2
nyc: 1
of: 17
on: 2
only: 1
other: 3
paul: 1
performance: 2
pete: 1
point: 1
produced: 2
public: 1
recommend: 1
release: 1
released: 1
return: 1
richard: 1
rock: 1
rod: 1
role: 1
roll: 1
said: 1
same: 1
show: 1
simplistic: 1
so

Each line tells you which word appeared, and how many times, so it's forming the full review, but now in vector form.

# Part C

In [155]:
# create the model (1 hidden layer with 100 neurons)
model = MLPClassifier(hidden_layer_sizes=(100,), max_iter=100, random_state=42)

# train the model
model.fit(X_train_vec, Y_train)

# predict on validation set
val_preds = model.predict(X_val_vec)

# check accuracy
val_acc = accuracy_score(Y_val, val_preds)
print("Validation accuracy:", val_acc)

Validation accuracy: 0.8836


88,36% is a solid accuracy, but let's try to tune it a bit more.

Now I will:
- increase the number of neurons to (200,)
- change the activation function to "tanh"
- increase max_iter to 200
- enable early stopping

Let's see if it will help our accuray. 🤓

In [159]:
model = MLPClassifier(
    hidden_layer_sizes=(200,),
    activation='tanh',
    max_iter=200,
    early_stopping=True,
    random_state=42
)

model.fit(X_train_vec, Y_train)
val_preds = model.predict(X_val_vec)
val_acc = accuracy_score(Y_val, val_preds)
print("Validation accuracy:", val_acc)

Validation accuracy: 0.882


It didn't really help the accuracy, it's a bit lower now but still pretty good.

Let's play with it a bit more.

In [180]:
model = MLPClassifier(
    hidden_layer_sizes=(150,),
    activation='relu',
    max_iter=200,
    early_stopping=True,
    random_state=42
)

model.fit(X_train_vec, Y_train)
val_preds = model.predict(X_val_vec)
val_acc = accuracy_score(Y_val, val_preds)
print("Validation accuracy:", val_acc)

Validation accuracy: 0.886


In this version, I changed the number of neurons to 150 (instead of 100 or 200).
I also kept activation='relu', increased max_iter to 200, and enabled early_stopping=True.
Validation accuracy is 88.6% which is pretty solid. :)

# Part D

Now when our model is trained and tuned, we will evaluate it on the test set.

This gives us a final estimate of how well it performs on data that the model haven't seen before.

In [186]:
# predict on the test set
test_preds = model.predict(X_test_vec)

# check accuracy
test_acc = accuracy_score(Y_test, test_preds)
print("Test accuracy:", test_acc)

Test accuracy: 0.8864


The accuracy (88.64%) is very close to the validation accuracy (88.6%), which means the model didn’t overfit and is performing consistently. :)

# Part E

Now as this is the last part, I will add some of my own reviews, vectorize them using the same **CountVectorizer**, and use my model to predict whether the review is positive or negative.

In [192]:
# my custom reviews
custom_reviews = [
    "This movie was absolutely amazing! I loved every second of it.",
    "Terrible film. It was a waste of time and the acting was awful.",
    "It was okay, not great but not the worst either.",
    "One of the best films I've seen in years. Truly inspiring!",
    "I couldn't even finish it, it was so boring and predictable."
]

# vectorize using the same vectorizer
custom_reviews_vec = vectorizer.transform(custom_reviews)

# predict sentiment
custom_preds = model.predict(custom_reviews_vec)

# show results
for review, pred in zip(custom_reviews, custom_preds):
    sentiment = "Positive" if pred == 1 else "Negative"
    print(f"Review: {review}\nPredicted Sentiment: {sentiment}\n")

Review: This movie was absolutely amazing! I loved every second of it.
Predicted Sentiment: Positive

Review: Terrible film. It was a waste of time and the acting was awful.
Predicted Sentiment: Negative

Review: It was okay, not great but not the worst either.
Predicted Sentiment: Negative

Review: One of the best films I've seen in years. Truly inspiring!
Predicted Sentiment: Positive

Review: I couldn't even finish it, it was so boring and predictable.
Predicted Sentiment: Negative

